In [2]:
import requests
url= "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)


In [3]:
text = response.text

# Write data to file
with open('shakespeare_data.txt', 'w', encoding='utf-8') as f:
    f.write(text)

# Read data from file
with open('shakespeare_data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Data read successfully! Total characters: {len(text)}")


Data read successfully! Total characters: 1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
#conbiinig all the special characters from the text
chars=sorted(list(set(text)))
print(''.join(chars))
vocab_size=len(chars)
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# defining key:value pair
stoi={ch:i for i, ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode= lambda alphabats:[stoi[c] for c in alphabats] # encoding the  list of string,in corresponding integers
decode = lambda list_of_integers: ''.join([itos[i] for i in list_of_integers])

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [7]:
import torch
print(torch.__version__)

2.10.0+cpu


In [8]:
#encoding the entire dataset 
import torch
data=torch.tensor(encode(text), dtype=torch.long)
print(data.shape,data.size)
#showing 1000 records of integers
print(data[:1000])


torch.Size([1115394]) <built-in method size of Tensor object at 0x000002951CE64AA0>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61, 

In [9]:
#splitting the data into validation and training
#now split into traininng
n=int(0.9*len(data)) #90 percent will train and 10 percent will be validate
train_data=data[:n]
val_data=data[n:]

In [10]:
# will pass the chunks of data 
block_size=8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
# this creating the prediction of next integer based on the context we provided
#from start till block size 
X_train=train_data[:block_size]
# from 1 till current + 1 
y_train=train_data[1:block_size+1]
for t in range(block_size):
    context=X_train[:t+1]
    target=y_train[t]
    print(f"when you see the {context} predict next sequence {target}")

when you see the tensor([18]) predict next sequence 47
when you see the tensor([18, 47]) predict next sequence 56
when you see the tensor([18, 47, 56]) predict next sequence 57
when you see the tensor([18, 47, 56, 57]) predict next sequence 58
when you see the tensor([18, 47, 56, 57, 58]) predict next sequence 1
when you see the tensor([18, 47, 56, 57, 58,  1]) predict next sequence 15
when you see the tensor([18, 47, 56, 57, 58,  1, 15]) predict next sequence 47
when you see the tensor([18, 47, 56, 57, 58,  1, 15, 47]) predict next sequence 58


In [12]:
torch.manual_seed(1337)

batch_size=4
block_size=8

def get_batch(split):
    data=train_data if split=='train' else val_data
    ix=torch.randint(len(data) - block_size,(batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb=get_batch('train')
print(xb)
print(xb.shape)
print('target')
print(yb)
print(yb.shape)
print("-----------")


for a in range(batch_size):
    for b in range(block_size):
        context=xb[a,:b+1]
        target=yb[a,:b]
        print(f"when the input is {context.tolist()} the target is :{target} ")

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
target
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])
-----------
when the input is [24] the target is :tensor([], dtype=torch.int64) 
when the input is [24, 43] the target is :tensor([43]) 
when the input is [24, 43, 58] the target is :tensor([43, 58]) 
when the input is [24, 43, 58, 5] the target is :tensor([43, 58,  5]) 
when the input is [24, 43, 58, 5, 57] the target is :tensor([43, 58,  5, 57]) 
when the input is [24, 43, 58, 5, 57, 1] the target is :tensor([43, 58,  5, 57,  1]) 
when the input is [24, 43, 58, 5, 57, 1, 46] the target is :tensor([43, 58,  5, 57,  1, 46]) 
when the input is [24, 43, 58, 5, 57, 1, 46, 43] the target is :tensor([43, 58,  5, 57

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLangModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding_table=nn.Embedding(vocab_size,vocab_size)
    def forward(self,idx,target=None):
        logits=self.embedding_table(idx)

        if target==None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            target=target.view(B*T)
            loss=F.cross_entropy(logits,target)
        return logits,loss
    def generate(self,idx,max_new_tokens):
        
        for _ in range(max_new_tokens):
            logits, loss=self(idx)
            logits=logits[:,-1,:]# (B,C)
            probs=F.softmax(logits, dim=1) #to get probab
            idx_next=torch.multinomial(probs,num_samples=1)#sample from distribution
            idx=torch.cat((idx,idx_next),dim=1)#append to the index with new prediction(B,T+1)
        return idx

In [14]:
m=BigramLangModel(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [15]:
# create a pytorch optimizer
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)


In [16]:
batch_size=32
#sample of batch
for steps in range(1000):
    xb,yb=get_batch('train')
    #evaluate loss
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    #backward propagation
    loss.backward()
    optimizer.step()

print(loss.item())

3.704136848449707


In [17]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV-ivvKcOvi-nQGX&q-YQbm dEM?px;Akr-IESq--wIWId
RFgXTpDUgM:CK$I!uo'IBT -
j?wfy fFr.&fiqtRS.ZttxGh' a!ogrn$zoZqbocL&yIffBDWNUboscuQqo.Fls,?,M?eZxHx?p?EV.mJiHqHnxT  bQpa;P fawiF$-QbWv&f:CVDCBfano,b?$Esev.?


In [18]:
# example using how matrix calculation can be used for weighted aggregation
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [19]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [20]:
xbow=torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        x_prev=x[b,:t+1]
        xbow[b,t]=torch.mean(x_prev,0)

In [21]:
# creating a self attention layer
torch.manual_seed(1337)
B,T,C= 4,8,32
x=torch.rand(B,T,C)

head_size=16
k_value=nn.Linear(C,head_size,bias=False)
q_query=nn.Linear(C,head_size,bias=False)
value=nn.Linear(C,head_size,bias=False)
k=k_value(x)
q=q_query(x)
wei=q @ k.transpose(-2,-1)

# it creates square of ones
tril=torch.tril(torch.ones(T,T))
## fill we used because future prediction will
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=-1)

v=value(x)

out=wei @ v
out.shap

AttributeError: 'Tensor' object has no attribute 'shap'

In [ ]:
k=torch.randn(B,T,head_size)
q=torch.randn(B,T,head_size)
wei=q @ k.transpose(-2,-1)**head_size**-0.5

In [ ]:
wei[0]

tensor([[nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan]])

In [22]:
k.var()

tensor(0.0679, grad_fn=<VarBackward0>)

In [ ]:
# creating a head class 
n_embed=32
dropout=0.0
n_head = 4
n_layer = 4

class Head(nn.Module):
    """ one head of self attention"""

    def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embed,head_size,bias=False)
        self.query=nn.Linear(n_embed,head_size,bias=False)
        self.value=nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

    def forward(self,x):
        B,T,C=x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x)
        #compute attention scores
        wei= q @ k.transpose(-2,1) ** C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei=F.softmax(wei,dim=1)
        wei=self.dropout(wei)
        v=self.value(x)
        out=wei @ v
        return out

class MultiheadAttention(nn.Module):
    """ multi head attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # These must be indented under __init__
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout) # Make sure 'dropout' variable is defined

    def forward(self, x):
       
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    
class feedforward(nn.Module):
        
        def __init__(self,n_embed):
            super().__init__()
            self.net=nn.Sequential(
                nn.Linear(n_embed, 4*n_embed),
                nn.ReLU(),\
                nn.Linear(4 *n_embed,n_embed),
                nn.Dropout(dropout)
            )
        def forward(self, x):
            return self.net(x)
    # This line takes the input 'x' and pushes it through 
    # the entire 'net' pipeline we built in __init__.
    
class Block(nn.Module):

    def __init__(self,n_embed,n_head):
            #n_embedd : number for dimension we want to expand and n_head:the num of head we like to divide
        super().__init__()
        head_size=n_embed / n_head # each head gets equal slice of embedding (512/8=64)
        self.sa=MultiheadAttention(n_head,head_size) # 8 heads look at input differently and share what they found
        self.ffwd=feedforward # each token processes what it learned from attention deeply
        self.ln1=nn.LayerNorm(n_embed) # stabilize numbers before feeding into attention
        self.ln2=nn.LayerNorm(n_embed) # stabilize numbers again before feeding into feedforward


    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # stabilize x → tokens talk to each other → dont forget original x
        x = x + self.ffwd(self.ln2(x)) # stabilize x → tokens think deeply → dont forget what attention learned
        return x                        # x is now smarter than when it came in

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)# turn each word/character into a 512dim vector
        self.position_embedding_table = nn.Embedding(block_size, n_embed) # turn each position(1st,2nd,3rd..) into a 512dim vector
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head) for _ in range(n_layer)]) # stack multiple transformer blocks on top of each other
        self.ln_f = nn.LayerNorm(n_embed) # stabilize numbers after all blocks are done
        self.lm_head = nn.Linear(n_embed, vocab_size)# convert final 512dims into scores for every word in vocabulary

    def forward(self, idx, targets=None):
        B, T = idx.shape  # B = number of sentences, T = number of tokens per sentence

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # convert token numbers → 512dim vectors (WHAT each token is)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # convert positions → 512dim vectors (WHERE each token is)
        x = tok_emb + pos_emb     # combine WHAT + WHERE so model understands order
        x = self.blocks(x)     # pass through all transformer blocks to deeply understand context
        x = self.ln_f(x)     # final stabilization before making predictions
        logits = self.lm_head(x)# convert deep understanding → scores for next word prediction

        if targets is None:
            loss = None # inference mode: just generating text, no need to calculate loss
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # flatten all predictions into one long list
            targets = targets.view(B*T) # flatten all real answers to match predictions
            loss = F.cross_entropy(logits, targets)  # calculate how wrong predictions are vs real answers

        return logits, loss  # return next word scores and how wrong we were
        






    def generate(self,idx,max_new_tokens):
        
        for _ in range(max_new_tokens):
            #crop index to the last block size
            idx_cond = idx[:, -block_size:]
            #gets the prediction
            logits, loss=self(idx_cond)
            #focus only the last time step
            logits=logits[:,-1,:]# (B,C)
            #apply softmax to get probab
            probs=F.softmax(logits, dim=1) #to get probab
            idx_next=torch.multinomial(probs,num_samples=1)#sample from distribution
            idx=torch.cat((idx,idx_next),dim=1)#append to the index with new prediction(B,T+1)
        return idx
    
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


        
